# **TracIn**

En este documento realizamos la técnica TracIn en nuestros datos, para cambiar de modelo predictivo a la que realizarla basta con cambiar las configuraciones inciales y el dataset que se carga.

**Necesario archivo tracIn_explainer.py** o en su defecto, la librería `glassbox_vit` instalada.

In [ ]:
from huggingface_hub import HfApi
from huggingface_hub import download_bucket_files, list_bucket_tree
from pathlib import Path
import os
from datasets import load_dataset, Image
import numpy as np
from PIL import Image as PILImage
from io import BytesIO
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor
import pandas as pd

from glassbox_vit.tracIn_explainer import TracInExplainer

In [ ]:
os.environ["HF_TOKEN"] = "mi_token"

token = os.environ.get("HF_TOKEN")
if not token:
    raise EnvironmentError("Define HF_TOKEN en el entorno antes de ejecutar esta celda")

api = HfApi(token=token)

bucket_id = "z72pepee/tfm-explicabilidad"

carpeta_objetivo = "Modelos_Entrenados/experimento_dataset1_microsoft/"

# Obtenemos la lista exacta de archivos dentro de esa carpeta
archivos_remotos = [
    item.path
    for item in list_bucket_tree(bucket_id, prefix=carpeta_objetivo)
    if item.type == "file" and "checkpoint-" in item.path
]

# Construimos la lista de tuplas que necesita download_bucket_files
# Formato: (ruta_en_el_bucket, ruta_en_tu_disco_local)
# Al poner (ruta, ruta) se guardarán conservando la misma estructura de carpetas.
archivos_a_descargar = [
    (ruta, ruta)
    for ruta in archivos_remotos
]

print(f"¡Se han encontrado {len(archivos_a_descargar)} archivos para descargar!")

# Descargamos los archivos en bloque
if archivos_a_descargar:
    download_bucket_files(
        bucket_id,
        files=archivos_a_descargar
    )
    print("¡Descarga completada con éxito!")
else:
    print("No se encontró nada para descargar (la carpeta no existe o está vacía).")



¡Se han encontrado 104 archivos para descargar!


¡Descarga completada con éxito!


In [ ]:
# Initialize the explainer pointing to your downloaded checkpoints
explainer = TracInExplainer(
    checkpoints_dir="Modelos_Entrenados/experimento_dataset1_microsoft/",
    learning_rate=5e-5,
    device="cuda"
)

data1 = load_dataset("z72pepee/skin-gan-review-baseline", "PAD-UFES", token="mi_token")


raw_train_data = data1["train"].cast_column("image", Image(decode=False))

# Create an empty list to store the file names
image_names = []

# Loop through the dataset very quickly
for idx, example in enumerate(raw_train_data):
    # Extract the file path directly
    file_path = example["image"]["path"]

    # Ensure the path exists. If it does, extract just the file name.
    if file_path is not None:
        final_name = os.path.basename(file_path)
    else:
        # Fallback if there is no path
        final_name = f"train_dataset_image_{idx}"

    image_names.append(final_name)

print(f"Extracted {len(image_names)} image names successfully!")
print(f"Example of the first 3 names: {image_names[:3]}")


# Load the processor (feature extractor) that matches ViT model
processor_path = "Modelos_Entrenados/experimento_dataset1_microsoft/checkpoint-130"
image_processor = AutoImageProcessor.from_pretrained(processor_path)

# Define a transformation function
def transform_images(examples):
    # Convert PIL images into PyTorch tensors (pixel_values)
    # .convert("RGB") ensures images have 3 channels (no grayscale or PNG alpha channels)
    images = [img.convert("RGB") for img in examples["image"]]
    inputs = image_processor(images, return_tensors="pt")

    # Keep the labels in the batch
    inputs["labels"] = examples["label"]
    return inputs

# Apply the transform on-the-fly to your training split
processed_train_dataset = data1["train"].with_transform(transform_images)

# Create the DataLoader using the PROCESSED dataset
# shuffle=False to keep the order identical to your names list
train_dataloader = DataLoader(processed_train_dataset, batch_size=16, shuffle=False)

# Pass your PyTorch DataLoader. This will take time
explainer.prepare_database(train_dataloader, image_names=image_names, save_path="base_datos_tracin.pt")

README.md:   0%|          | 0.00/35.4k [00:00<?, ?B/s]

PAD-UFES/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  159MB            

PAD-UFES/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

PAD-UFES/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 9.15MB            

PAD-UFES/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

PAD-UFES/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 8.87MB            

... (29 outputs left out)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Processing checkpoint-1690...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Database successfully saved to base_datos_tracin.pt.


In [ ]:
# Extract the names of the test images (just like we did for train)
raw_test_data = data1["test"].cast_column("image", Image(decode=False))
test_image_names = []
for idx, example in enumerate(raw_test_data):
    file_path = example["image"]["path"]
    final_name = os.path.basename(file_path) if file_path else f"test_img_{idx}"
    test_image_names.append(final_name)

# Apply the image processor (the transform_images function we defined earlier)
processed_test_dataset = data1["test"].with_transform(transform_images)

# Create the DataLoader for the test set
# VERY IMPORTANT: shuffle=False again
test_dataloader = DataLoader(processed_test_dataset, batch_size=16, shuffle=False)

# Run the bulk evaluation
results = explainer.evaluate_test_dataset(
    test_dataloader=test_dataloader,
    test_image_names=test_image_names,
    top_k=5,
    save_path="resultados_test_tracin.json"
)

Step 1: Extracting test gradients across 13 checkpoints...
Loading checkpoint-130...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading checkpoint-260...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading checkpoint-390...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

... (17 outputs left out)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading checkpoint-1690...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


Step 2: Computing influence scores against training database (This is fast!)...

Done! Results for 115 test images saved to resultados_test_tracin.json


In [ ]:
csv_rows = []

for test_img_name, data in results.items():
    # Create a base row with the test image name
    row = {"test_image_name": test_img_name}

    # Unpack the 5 proponents into flat columns
    for i, (train_name, score) in enumerate(data["proponents"]):
        row[f"proponent_{i+1}_name"] = train_name
        row[f"proponent_{i+1}_score"] = score

    # Unpack the 5 opponents into flat columns
    for i, (train_name, score) in enumerate(data["opponents"]):
        row[f"opponent_{i+1}_name"] = train_name
        row[f"opponent_{i+1}_score"] = score

    csv_rows.append(row)

# Convert the list of rows into a Pandas DataFrame
df_results = pd.DataFrame(csv_rows)

# Save to CSV
csv_filename = "resultados_test_tracin.csv"
df_results.to_csv(csv_filename, index=False)

print(f"Success! Data flattened and saved to {csv_filename}")
# Display the first few rows to verify
print(df_results.head())

Success! Data flattened and saved to resultados_test_tracin.csv
         test_image_name      proponent_1_name  proponent_1_score  \
0    PAT_1017_97_577.png  PAT_715_1338_761.png           0.048354   
1   PAT_1172_622_420.png   PAT_1128_497_29.png           0.060380   
2   PAT_1259_892_793.png   PAT_86_1082_220.png           0.118728   
3  PAT_1300_1056_532.png  PAT_851_4519_488.png           0.177683   
4  PAT_1374_1285_735.png  PAT_1035_165_800.png           0.027045   

       proponent_2_name  proponent_2_score       proponent_3_name  \
0    PAT_412_818_61.png           0.046796    PAT_498_944_488.png   
1  PAT_817_4471_179.png           0.059682     PAT_65_101_847.png   
2   PAT_309_663_812.png           0.116965    PAT_155_238_413.png   
3   PAT_202_308_721.png           0.153414     PAT_38_1003_68.png   
4  PAT_2036_4280_90.png           0.026100  PAT_1931_3879_727.png   

   proponent_3_score       proponent_4_name  proponent_4_score  \
0           0.045700   PAT_757_1431_907.

In [ ]:
archivos_a_subir = [(str(Path("resultados_test_tracin.json")),"dataset1_microsoft/tracin/resultados_test_tracin.json")
,(str(Path("resultados_test_tracin.csv")),"dataset1_microsoft/tracin/resultados_test_tracin.csv"),
                    (str(Path("base_datos_tracin.pt")),"dataset1_microsoft/tracin/base_datos_tracin.pt")]


# Subimos los archivos
try:
    api.batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print("Elementos subidos exitosamente al bucket")
except Exception as e:
    print(f"Error: {e}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tent/base_datos_tracin.pt:   0%|          |  558kB /  133MB            

  ...esultados_test_tracin.csv:   1%|          |   429B / 50.3kB            

  ...sultados_test_tracin.json:   1%|          | 1.14kB /  133kB            

Elementos subidos exitosamente al bucket
